# Recorded-result reproduction — CPU only

This notebook audits existing JSON evidence. It does not train, load a policy, run simulation, or modify the source project. Run A → B → C. The English README explains scope and limitations.

## A. Mount the existing project
Choose a CPU runtime. If using a different Drive account or folder, edit PROJECT.

In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/embodied-smolvla")
if not PROJECT.is_dir():
    raise FileNotFoundError(f"Existing project not found: {PROJECT}")
print("Source:", PROJECT)


## B. Load the embedded calculation entry point
Standard-library code only. No dependency installation or model calls.

In [ ]:
import types
analysis = types.ModuleType("smolvla_closure_analysis")
exec(compile('"""Recalculate committed historical results. Standard library only; no model execution."""\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\n\nSERIES = {\n    \'rq1\': (\'paired_task0_states0_19_v1\', (\'1\', \'5\')),\n    \'original\': (\'occlusion_center_task0_states0_19_v1\', (\'clean\', \'occluded\')),\n    \'round1\': (\'finetuned_task0_clean_occluded_v1\', (\'clean\', \'occluded\')),\n    \'round2\': (\'low_lr_fixed150_task0_clean_occluded_v1\', (\'clean\', \'occluded\')),\n}\n\ndef require(condition, message):\n    if not condition:\n        raise ValueError(message)\n\ndef read(path):\n    return json.loads(path.read_text(encoding=\'utf-8\'))\n\ndef audit_series(project, label):\n    folder, conditions = SERIES[label]\n    root = project / \'results\' / folder\n    plan = read(root / \'plan.json\')\n    summary = read(root / \'summary.json\')\n    plan_hash = hashlib.sha256(json.dumps(plan, sort_keys=True).encode()).hexdigest()\n    require(plan_hash == summary[\'plan_hash\'], f\'{label}: plan hash mismatch\')\n    require(plan[\'state_ids\'] == list(range(20)), f\'{label}: unexpected state list\')\n    require(plan[\'suite\'] == \'libero_spatial\' and plan[\'task_id\'] == 0,\n            f\'{label}: unexpected task\')\n    require(plan[\'seed_base\'] == 42, f\'{label}: unexpected base seed\')\n    paths = sorted(root.glob(\'state_*/*/result.json\'))\n    require(len(paths) == 40, f\'{label}: expected 40 committed records, found {len(paths)}\')\n    rows, index = [], {}\n    for path in paths:\n        data = read(path)\n        state = data[\'state_id\']\n        condition = str(data[\'n_action_steps\']) if label == \'rq1\' else data[\'condition\']\n        key = (state, condition)\n        require(key not in index, f\'{label}: duplicate {key}\')\n        require(type(state) is int and 0 <= state < 20 and condition in conditions,\n                f\'{label}: invalid state/condition {key}\')\n        expected_folder = \'k\' + condition if label == \'rq1\' else condition\n        require(path.relative_to(root).as_posix() == f\'state_{state:03d}/{expected_folder}/result.json\',\n                f\'{label}: filename disagrees with record {path}\')\n        require(data[\'status\'] == \'completed\' and data[\'plan_hash\'] == plan_hash,\n                f\'{label}: uncommitted or wrong-plan result {key}\')\n        require(data[\'seed\'] == 42 + state, f\'{label}: seed mismatch {key}\')\n        require(type(data[\'success\']) is bool, f\'{label}: non-Boolean success {key}\')\n        require(type(data[\'control_steps\']) is int and 1 <= data[\'control_steps\'] <= 280,\n                f\'{label}: invalid action count {key}\')\n        require(isinstance(data[\'eval_seconds\'], (int, float)) and\n                math.isfinite(data[\'eval_seconds\']) and data[\'eval_seconds\'] >= 0,\n                f\'{label}: invalid elapsed time {key}\')\n        require(data[\'n_action_steps\'] == (int(condition) if label == \'rq1\' else 1),\n                f\'{label}: action-step setting mismatch {key}\')\n        index[key] = data\n        rows.append(dict(series=label, state=state, condition=condition,\n                         goal_success=data[\'success\'], control_steps=data[\'control_steps\'],\n                         eval_seconds=data[\'eval_seconds\'], result_path=str(path.relative_to(project)),\n                         sha256=hashlib.sha256(path.read_bytes()).hexdigest()))\n    require(set(index) == {(s, c) for s in range(20) for c in conditions},\n            f\'{label}: incomplete pairing\')\n    aggregates = []\n    for condition in conditions:\n        selected = [index[s, condition] for s in range(20)]\n        wins = sum(x[\'success\'] for x in selected)\n        seconds = sum(x[\'eval_seconds\'] for x in selected)\n        saved = summary[\'groups\'][condition]\n        require(saved[\'successes\'] == wins and saved[\'completed\'] == 20 and\n                saved[\'paired_n\'] == 20 and abs(saved[\'paired_success_rate\'] - wins/20) < 1e-12 and\n                abs(saved[\'paired_eval_seconds\'] - seconds) < 1e-6,\n                f\'{label}/{condition}: saved summary mismatch\')\n        aggregates.append(dict(series=label, condition=condition, episodes=20,\n                               goal_successes=wins, goal_rate=wins/20,\n                               control_steps=sum(x[\'control_steps\'] for x in selected),\n                               eval_seconds=seconds))\n    require(summary[\'total_completed\'] == summary[\'total_requested\'] == 40 and\n            summary[\'paired_state_ids\'] == list(range(20)), f\'{label}: summary coverage mismatch\')\n    return index, rows, aggregates\n\ndef write_csv(path, rows):\n    if not rows:\n        return\n    with path.open(\'w\', newline=\'\', encoding=\'utf-8\') as stream:\n        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))\n        writer.writeheader()\n        writer.writerows(rows)\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--project\', type=Path, required=True, help=\'Existing embodied-smolvla folder\')\n    parser.add_argument(\'--output\', type=Path, required=True, help=\'New, empty output directory\')\n    parser.add_argument(\'--series\', nargs=\'+\', choices=list(SERIES), default=list(SERIES))\n    args = parser.parse_args(argv)\n    require(len(args.series) == len(set(args.series)), \'Duplicate series selection\')\n    project, output = args.project.resolve(), args.output.resolve()\n    require(not output.is_relative_to(project), \'Output must be outside the source project\')\n    require(not output.exists() or not any(output.iterdir()), \'Output must be empty or absent\')\n    indices, episodes, aggregates = {}, [], []\n    for label in args.series:\n        index, rows, totals = audit_series(project, label)\n        indices[label] = index\n        episodes.extend(rows)\n        aggregates.extend(totals)\n    transitions = []\n    if \'original\' in indices:\n        for candidate in (\'round1\', \'round2\'):\n            if candidate not in indices:\n                continue\n            for condition in (\'clean\', \'occluded\'):\n                for state in range(20):\n                    before = indices[\'original\'][state, condition][\'success\']\n                    after = indices[candidate][state, condition][\'success\']\n                    transitions.append(dict(candidate=candidate, state=state, condition=condition,\n                        original_goal=before, candidate_goal=after,\n                        change=\'unchanged\' if before == after else (\'repair\' if after else \'regression\')))\n    if all(k in indices for k in (\'original\', \'round1\', \'round2\')):\n        for state in (5, 14):\n            require([indices[k][state, \'clean\'][\'success\'] for k in (\'original\', \'round1\', \'round2\')]\n                    == [True, True, False], f\'Corrected case {state:03d} no longer agrees with sources\')\n    output.mkdir(parents=True, exist_ok=True)\n    write_csv(output/\'episode_ledger.csv\', episodes)\n    write_csv(output/\'summary.csv\', aggregates)\n    write_csv(output/\'transitions.csv\', transitions)\n    result = dict(status=\'passed\', scope=\'recorded-result recalculation; no inference or training\',\n                  series=args.series, committed_episodes=len(episodes), aggregates=aggregates)\n    (output/\'audit.json\').write_text(json.dumps(result, indent=2), encoding=\'utf-8\')\n    print(json.dumps(result, indent=2))\n    return result\n\nif __name__ == \'__main__\':\n    main()\n', "reproduce_results.py", "exec"), analysis.__dict__)
print("Analysis entry point loaded; no experiment executed.")


## C. Recalculate and validate all 160 committed records
Outputs go to a new temporary /content directory outside Drive. The audit stops if a record is missing or inconsistent. It never fills gaps by running a model.

In [ ]:
import tempfile
OUTPUT = Path(tempfile.mkdtemp(prefix="smolvla_closure_tables_", dir="/content"))
result = analysis.main(["--project", str(PROJECT), "--output", str(OUTPUT)])
print("Output directory:", OUTPUT)
print((OUTPUT / "summary.csv").read_text())


## Scope
This reproduces historical calculations, not policy behavior. Later finishing traces, full training reproduction and final report assembly are separate closure tasks. Source results and notebooks remain unchanged.

## README

# SmolVLA: Single-Task Evaluation and Adaptation Study

This project studies action execution steps, central visual occlusion, and limited-demonstration adaptation of a pretrained SmolVLA policy in LeRobot/LIBERO. The task is to pick up the black bowl between the plate and ramekin and place it on the plate (`libero_spatial`, task 0).

**Closure status, 25 October 2026:** the recorded-result recalculation entry point is ready. The final research report and a fully assembled training/simulation reproduction environment are not yet complete. No verified better autonomous fine-tuned policy was obtained. The present adaptation branch is stopped.

## What this entry point does

`reproduce_results.py` reads the original committed JSON records, checks coverage, plans, seeds and saved summaries, and generates English CSV tables and a JSON audit. It uses only the Python standard library. It does not import a model, install GPU dependencies, run simulation, change model weights, alter source results or judge videos.

`00_smolvla_reproduce_recorded_results_cpu.ipynb` embeds the same code and is the standalone Colab entry point. You do not need to run both the notebook and the script.

This is **reproduction of reported calculations from recorded evidence**, not a new independent reproduction of policy behavior. New rollouts and retraining are separate, resource-intensive operations and are not required for closure.

## Quick start: Colab CPU

1. Open `00_smolvla_reproduce_recorded_results_cpu.ipynb` in Colab with a CPU runtime.
2. Run A to mount the Google Drive account holding `embodied-smolvla`.
3. Run B to load the embedded analysis code.
4. Run C to audit the four historical series and create tables under a new `/content/smolvla_closure_tables_*` directory.

The notebook does not automatically launch training or simulation. Results are written to temporary Colab storage, outside the source Drive project. Copy them elsewhere if you want to retain them after the runtime ends. Missing or inconsistent inputs stop the audit; nothing is silently regenerated.

## Quick start: local Python

Use Python 3.10 or later. No additional packages are needed:

```bash
python reproduce_results.py --project /path/to/embodied-smolvla --output /path/to/new-closure-tables
```

The output directory must be empty or absent and outside the source project. For a deliberately partial audit, use `--series original round1 round2`; partial coverage is explicitly recorded in `audit.json`.

## Required evidence layout

All paths are beneath `embodied-smolvla/results/`:

| Series | Directory | Per-state conditions |
|---|---|---|
| RQ1 | `paired_task0_states0_19_v1` | `k1`, `k5` |
| Original visual baseline | `occlusion_center_task0_states0_19_v1` | `clean`, `occluded` |
| Fine-tuning round 1 | `finetuned_task0_clean_occluded_v1` | `clean`, `occluded` |
| Fine-tuning round 2 | `low_lr_fixed150_task0_clean_occluded_v1` | `clean`, `occluded` |

Each directory needs `plan.json`, `summary.json`, and 40 committed files named `state_000/<condition>/result.json` through `state_019/<condition>/result.json`. Attempts, videos and model weights are unnecessary for this calculation entry point; retain them separately for provenance and physical-behavior analysis.

## Reported results and interpretation

| Comparison | Historical benchmark-goal result |
|---|---|
| RQ1, execution steps 1 vs 5 | 16/20 vs 15/20; cumulative evaluation time 3385.618221 vs 1959.911733 seconds |
| Original, clean vs occluded | 16/20 vs 12/20 |
| Round 1, clean / occluded | 13/20 / 10/20; total 23/40 |
| Round 2, clean / occluded | 14/20 / 14/20; total 28/40 |

These are paired development states. Forty condition-rollouts are not forty independent initial states. The observed 42.11% lower RQ1 evaluation time is not a pure inference-speed measurement; episode lengths differ. Direct chunk-generation call counts were not stored in those result records.

Historical success means triggering the benchmark goal under the early stopping protocol. It does not establish genuine release, safe withdrawal, no re-contact or two seconds of stability. Scripted finishing demonstrations are not autonomous policy success.

**Corrected case attribution:** 005 clean and 014 clean succeed in the original and first-round records, and fail in round 2. Rightward tilt (005) and rightward translation (014) without lift belong to round 2. Notebook 08 explicitly establishes this; the user's notebook logs did not contain the conflict previously alleged during consolidation.

## Output files

| Output | Contents |
|---|---|
| `summary.csv` | Per-series goal counts, rates, action totals and elapsed time |
| `episode_ledger.csv` | Each committed record, relative source path and source-byte SHA-256 |
| `transitions.csv` | Original-to-candidate paired repairs and regressions |
| `audit.json` | Explicit pass status, scope and selected series |

No output is a new success label or video annotation. Exact input rows support the recomputed aggregates. Source plan hashes use the historical sorted-key JSON serialization.

## Minimal historical workflow map

Historical scripts are preserved in the Drive project. This is a dependency map, not an instruction to run all notebooks or proof that a clean runtime has been retested.

| Purpose | Source entry point | Required evidence/dependencies |
|---|---|---|
| RQ1 original evaluation | project `03_paired_eval_v1.py` | Locked original model, LIBERO assets, environment and plan |
| Visual-occlusion baseline | project `04_occlusion_center_v1.py` | RQ1 environment/compatibility records and mask protocol |
| Round-1 training | `finetune/07_task0_finetune_v1.py` | Prepared original demonstrations, split, original processors and training adapter |
| Round-1 evaluation | [Notebook 04](https://colab.research.google.com/drive/1vakHYZiaUQ4iO_Cx_tL_v9jTCbxYlDDo), project `08_finetuned_eval_v1.py` | Selected update-150 export and original visual baseline |
| Round-2 training | `finetune/10_low_lr_fixed150_v1.py` | Same demonstration split and locked original start; fixed update 150 |
| Round-2 evaluation | [Notebook 08](https://colab.research.google.com/drive/1y9bsPROi0UyLWLWpPHIqtEmLH8IMFCFE), project `12_low_lr_fixed150_eval_v1.py` | Round-2 export; reused original/round-1 comparisons |
| Later finishing work | notebooks 11–22, selected sources only | Separate scripted-data, training, diagnostic and policy-evaluation stages |

Script filename prefixes and notebook numbers differ. Minimal end-to-end extraction must resolve the training adapter, model snapshots, assets and initialization before any rerun. It is still pending; this package does not pretend those dependencies are bundled.

Key historical environment: Python 3.12.3, LeRobot 0.6.1, PyTorch 2.11.0+cu128, hf-libero 0.1.4, robosuite 1.4.0, NumPy 2.2.6, Tesla T4. These are recorded simulation/training versions; the CPU calculation script does not require that stack.

## Final delivery and remaining work

Retain necessary source/configuration files, committed result evidence, training provenance, and indexed success/failure/regression videos. Keep duplicates, installation output and exploratory notebooks in a historical backup rather than exposing all 22 as reproduction steps. No Drive source files have been deleted or reorganized.

The next deliverable is the English research report. Its later finishing section still needs the selected trace/diagnostic evidence linked; this CPU entry point covers the early 160-rollout main comparisons only. A public repository, a clean GPU rerun and a publication-quality scientific claim are not implied by this package.

## Validation of this calculation entry point

Locally tested on the existing 120-record original/round-1/round-2 evidence archive: all saved counts, rates and elapsed-time sums matched; corrected cases 005/014 passed. A deliberately changed seed was rejected. The notebook JSON parsed and every code cell compiled successfully; full nbformat schema validation was unavailable in this environment. The RQ1 branch has not been execution-tested with its 40 source records in this packaging pass; the earlier evidence audit independently verified those records. The Colab mount flow has not been run here.
